# 🤖 Motor de Recomendação Híbrido
**Instacart Market Basket Analysis · Etapa 5**

Combina os sinais gerados nas etapas anteriores em um único motor de recomendação:

| Sinal | Origem | Peso sugerido |
|---|---|---|
| Coocorrência (Market Basket) | Notebook 03 | 0.4 |
| Similaridade Cosseno | Notebook 04 | 0.3 |
| Popularidade / Recompra | Notebook 02 | 0.2 |
| Embedding (PyTorch) | Notebook 06 | 0.1 (se disponível) |

O motor gera recomendações **produto → produto** e também **usuário → produtos**, com avaliação via Precision@K, Recall@K, MAP@K e Coverage.

---

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="viridis")

## 📂 1. Carregamento dos Artefatos das Etapas Anteriores

In [ ]:
import gc

DATA_PATH      = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")

ORDER_PRODUCTS_PRIOR_PATH = DATA_PATH / "order_products__prior.csv"
ITEM_SIM_TOP50_PATH = PROCESSED_PATH / "item_item_similarity_top50.npz"
ITEM_SIM_FULL_PATH  = PROCESSED_PATH / "item_item_similarity.npz"

# Limpa variáveis grandes de execuções anteriores
for var in [
    "order_products_prior", "order_products_train", "orders",
    "item_item_similarity", "eval_df", "eval_sample",
    "last_prior_products", "train_products", "recommender"
]:
    if var in globals():
        del globals()[var]

gc.collect()

# Artefatos processados
products               = pd.read_parquet(PROCESSED_PATH / "product_features.parquet")
market_basket_rules     = pd.read_parquet(PROCESSED_PATH / "market_basket_rules.parquet")
product_index_mapping   = pd.read_parquet(PROCESSED_PATH / "product_index_mapping.parquet")

# Carregamento leve das tabelas CSV
orders = pd.read_csv(
    DATA_PATH / "orders.csv",
    usecols=[
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
    ],
    dtype={
        "order_id": "int32",
        "user_id": "int32",
        "eval_set": "category",
        "order_number": "int16",
        "order_dow": "int8",
        "order_hour_of_day": "int8",
        "days_since_prior_order": "float32",
    },
)

order_products_train = pd.read_csv(
    DATA_PATH / "order_products__train.csv",
    usecols=["order_id", "product_id", "add_to_cart_order", "reordered"],
    dtype={
        "order_id": "int32",
        "product_id": "int32",
        "add_to_cart_order": "int16",
        "reordered": "int8",
    },
)

# Importante: NÃO carregamos order_products__prior.csv inteiro.
# Ele será lido em chunks apenas na etapa de avaliação.
order_products_prior = None

# Similaridade: carrega somente a versão reduzida Top 50, se existir.
# A matriz completa item_item_similarity.npz é grande demais para este notebook.
if ITEM_SIM_TOP50_PATH.exists():
    item_item_similarity = sp.load_npz(ITEM_SIM_TOP50_PATH).tocsr()
    print("✅ Matriz de similaridade Top 50 carregada")
else:
    item_item_similarity = None
    print("⚠️ Matriz item_item_similarity_top50.npz não encontrada")
    print("   O recomendador seguirá com Market Basket + Popularidade.")

product_to_idx = dict(
    zip(
        product_index_mapping["product_id"].astype("int32"),
        product_index_mapping["matrix_index"].astype("int32"),
    )
)

idx_to_product = {int(v): int(k) for k, v in product_to_idx.items()}

product_name_map = dict(
    zip(
        products["product_id"].astype("int32"),
        products["product_name"],
    )
)

print("\n✅ Artefatos carregados:")
print(f"   · products:              {products.shape}")
print(f"   · market_basket_rules:   {market_basket_rules.shape}")
print(f"   · product_index_mapping: {product_index_mapping.shape}")
print(f"   · orders:                {orders.shape}")
print(f"   · order_products_train:  {order_products_train.shape}")

if item_item_similarity is not None:
    print(f"   · item_item_similarity:  {item_item_similarity.shape}")
else:
    print("   · item_item_similarity:  não carregada")
print("   · order_products_prior:  leitura em chunks")


## ⚖️ 2. Normalização dos Scores

Cada sinal está em uma escala diferente — normalizamos todos para [0, 1] antes de combinar.

In [ ]:
def min_max_normalize(series: pd.Series) -> pd.Series:
    """Normaliza uma série para o intervalo [0, 1]."""
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

# Normaliza recommendation_score do Market Basket
market_basket_rules["score_norm"] = (
    market_basket_rules
    .groupby("product_a_id")["recommendation_score"]
    .transform(min_max_normalize)
)

# Normaliza popularidade e recompra (sinal de fallback)
products["popularity_norm"]   = min_max_normalize(np.log1p(products["total_purchases"]))
products["reorder_rate_norm"] = products["reorder_rate"]  # já está em [0,1]

print("✅ Scores normalizados")
market_basket_rules[["product_a_id","product_b_id","recommendation_score","score_norm"]].head()

## 🧩 3. Classe `HybridRecommender`

In [ ]:
class HybridRecommender:
    """
    Motor de recomendação híbrido que combina:
      - Market Basket
      - Similaridade Cosseno, se disponível
      - Popularidade / Taxa de recompra como fallback
    """

    def __init__(
        self,
        mb_rules: pd.DataFrame,
        item_sim=None,
        product_to_idx: dict | None = None,
        idx_to_product: dict | None = None,
        products: pd.DataFrame | None = None,
        weights: dict | None = None,
    ):
        self.mb_rules = mb_rules.copy()
        self.item_sim = item_sim
        self.product_to_idx = product_to_idx or {}
        self.idx_to_product = idx_to_product or {}

        if products is None:
            raise ValueError("O DataFrame products precisa ser informado.")

        self.products = products.copy()
        self.products["product_id"] = self.products["product_id"].astype("int32")
        self.products = self.products.set_index("product_id")

        if "popularity_norm" not in self.products.columns:
            self.products["popularity_norm"] = 1.0

        if weights is not None:
            self.weights = weights
        elif self.item_sim is None:
            self.weights = {"market_basket": 0.75, "similarity": 0.00, "popularity": 0.25}
        else:
            self.weights = {"market_basket": 0.45, "similarity": 0.35, "popularity": 0.20}

        self._prepare_market_basket_rules()

    def _prepare_market_basket_rules(self):
        """Prepara índice rápido produto A → produtos B."""
        required = {"product_a_id", "product_b_id"}
        missing = required - set(self.mb_rules.columns)

        if missing:
            raise ValueError(
                f"As regras de Market Basket precisam das colunas {required}. "
                f"Colunas encontradas: {self.mb_rules.columns.tolist()}"
            )

        self.mb_rules["product_a_id"] = self.mb_rules["product_a_id"].astype("int32")
        self.mb_rules["product_b_id"] = self.mb_rules["product_b_id"].astype("int32")

        if "score_norm" not in self.mb_rules.columns:
            if "recommendation_score" in self.mb_rules.columns:
                score_col = "recommendation_score"
            elif "lift" in self.mb_rules.columns:
                score_col = "lift"
            elif "confidence" in self.mb_rules.columns:
                score_col = "confidence"
            else:
                score_col = None

            if score_col:
                self.mb_rules["score_norm"] = (
                    self.mb_rules
                    .groupby("product_a_id")[score_col]
                    .transform(min_max_normalize)
                )
            else:
                self.mb_rules["score_norm"] = 1.0

        self._mb_index = (
            self.mb_rules
            .groupby("product_a_id")
            .apply(lambda g: dict(zip(g["product_b_id"], g["score_norm"])))
            .to_dict()
        )

    def _market_basket_scores(self, product_id: int) -> dict:
        if product_id is None:
            return {}
        return self._mb_index.get(int(product_id), {})

    def _similarity_scores(self, product_id: int, top_k: int = 50) -> dict:
        """
        Retorna os produtos mais similares.
        Se a matriz de similaridade não estiver carregada, retorna vazio.
        """
        if self.item_sim is None or product_id is None:
            return {}

        product_id = int(product_id)

        if product_id not in self.product_to_idx:
            return {}

        idx = int(self.product_to_idx[product_id])

        sims = self.item_sim[idx].toarray().ravel()

        if len(sims) == 0:
            return {}

        sims[idx] = -1

        k = min(top_k, len(sims) - 1)

        if k <= 0:
            return {}

        top_idx = np.argpartition(-sims, k - 1)[:k]
        top_idx = top_idx[np.argsort(-sims[top_idx])]

        return {
            int(self.idx_to_product[i]): float(sims[i])
            for i in top_idx
            if sims[i] > 0 and i in self.idx_to_product
        }

    def recommend_for_product(self, product_id: int, top_n: int = 10) -> pd.DataFrame:
        """Recomendações produto → produto combinando os sinais disponíveis."""
        if product_id is None:
            return pd.DataFrame(
                columns=["product_id", "product_name", "mb_score", "sim_score", "pop_score", "final_score"]
            )

        product_id = int(product_id)

        mb_scores = self._market_basket_scores(product_id)
        sim_scores = self._similarity_scores(product_id)

        candidates = set(mb_scores.keys()) | set(sim_scores.keys())

        if not candidates:
            return self._popularity_fallback(top_n, exclude={product_id})

        rows = []

        for pid in candidates:
            pid = int(pid)

            mb_s = float(mb_scores.get(pid, 0.0))
            sim_s = float(sim_scores.get(pid, 0.0))

            if pid in self.products.index:
                pop_s = float(self.products.loc[pid, "popularity_norm"])
                product_name = self.products.loc[pid, "product_name"]
            else:
                pop_s = 0.0
                product_name = None

            final_score = (
                self.weights["market_basket"] * mb_s
                + self.weights["similarity"] * sim_s
                + self.weights["popularity"] * pop_s
            )

            rows.append({
                "product_id": pid,
                "product_name": product_name,
                "mb_score": mb_s,
                "sim_score": sim_s,
                "pop_score": pop_s,
                "final_score": final_score,
            })

        return (
            pd.DataFrame(rows)
            .sort_values("final_score", ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )

    def recommend_for_user(self, user_product_history: list[int], top_n: int = 10) -> pd.DataFrame:
        """Recomendações usuário → produtos, agregando scores do histórico."""
        if not user_product_history:
            return self._popularity_fallback(top_n)

        user_product_history = [int(pid) for pid in user_product_history if pid is not None]

        agg_scores = {}

        for pid in user_product_history:
            recs = self.recommend_for_product(pid, top_n=30)

            for _, row in recs.iterrows():
                rec_pid = int(row["product_id"])
                agg_scores[rec_pid] = agg_scores.get(rec_pid, 0.0) + float(row["final_score"])

        for pid in user_product_history:
            agg_scores.pop(pid, None)

        if not agg_scores:
            return self._popularity_fallback(top_n, exclude=set(user_product_history))

        rows = []

        for pid, score in agg_scores.items():
            product_name = self.products.loc[pid, "product_name"] if pid in self.products.index else None

            rows.append({
                "product_id": pid,
                "product_name": product_name,
                "final_score": score,
            })

        return (
            pd.DataFrame(rows)
            .sort_values("final_score", ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )

    def _popularity_fallback(self, top_n: int, exclude: set | None = None) -> pd.DataFrame:
        """Fallback baseado em popularidade, mantendo as colunas esperadas."""
        exclude = exclude or set()

        pop = (
            self.products[~self.products.index.isin(exclude)]
            .sort_values("popularity_norm", ascending=False)
            .head(top_n)
            .reset_index()
        )

        result = pop[["product_id", "product_name", "popularity_norm"]].copy()
        result["mb_score"] = 0.0
        result["sim_score"] = 0.0
        result["pop_score"] = result["popularity_norm"].astype(float)
        result["final_score"] = result["pop_score"] * self.weights.get("popularity", 1.0)

        return result[["product_id", "product_name", "mb_score", "sim_score", "pop_score", "final_score"]]


recommender = HybridRecommender(
    mb_rules=market_basket_rules,
    item_sim=item_item_similarity,
    product_to_idx=product_to_idx,
    idx_to_product=idx_to_product,
    products=products,
)

print("✅ HybridRecommender inicializado")

if_sim = "ativa" if item_item_similarity is not None else "desativada"
print(f"🔎 Similaridade cosseno: {if_sim}")


## 🧪 4. Testes do Motor

In [ ]:
def find_product_id(name: str) -> int | None:
    """Busca o product_id por nome exato ou parcial."""
    q = name.lower().strip()

    names = products["product_name"].str.lower().str.strip()

    exact = products[names == q]

    if not exact.empty:
        return int(exact.iloc[0]["product_id"])

    partial = products[names.str.contains(q, regex=False, na=False)]

    if not partial.empty:
        return int(partial.iloc[0]["product_id"])

    return None


pid = find_product_id("Organic Strawberries")

if pid is None:
    print("❌ Produto não encontrado")
else:
    print(f"✅ Produto encontrado: {pid} — {product_name_map.get(pid)}")
    display(recommender.recommend_for_product(pid, top_n=10))


In [ ]:
# Simula um usuário que comprou esses 3 produtos
history_names = ["Banana", "Organic Whole Milk", "Organic Avocado"]

history_ids = [find_product_id(n) for n in history_names]
history_ids = [h for h in history_ids if h is not None]

print(f"🛒 Histórico simulado: {history_names}")
print(f"🔎 IDs encontrados: {history_ids}")

display(recommender.recommend_for_user(history_ids, top_n=10))


## 📊 5. Visualização da Composição dos Scores

In [ ]:
pid = find_product_id("Organic Strawberries")
recs = recommender.recommend_for_product(pid, top_n=10)

if recs.empty:
    print("⚠️ Nenhuma recomendação encontrada para visualizar.")
else:
    fig, ax = plt.subplots(figsize=(12, 6))

    labels = recs["product_name"].fillna("Produto sem nome").str[:25]

    ax.barh(
        labels[::-1],
        recs["mb_score"][::-1],
        label="Market Basket",
        color=sns.color_palette("viridis", 3)[0],
    )

    ax.barh(
        labels[::-1],
        recs["sim_score"][::-1],
        left=recs["mb_score"][::-1],
        label="Similaridade",
        color=sns.color_palette("viridis", 3)[1],
    )

    ax.barh(
        labels[::-1],
        recs["pop_score"][::-1],
        left=(recs["mb_score"] + recs["sim_score"])[::-1],
        label="Popularidade",
        color=sns.color_palette("viridis", 3)[2],
    )

    ax.set_title("🧩 Composição dos Scores por Sinal — Top 10 Recomendações", fontsize=13, fontweight="bold")
    ax.set_xlabel("Score por sinal")
    ax.legend()

    plt.tight_layout()
    plt.show()


## 📐 6. Avaliação Completa do Motor

Métricas implementadas:
- **Precision@K** — proporção de recomendações relevantes
- **Recall@K** — proporção de itens relevantes recuperados
- **MAP@K** (Mean Average Precision) — considera a ordem das recomendações
- **Coverage** — proporção do catálogo que o modelo consegue recomendar

In [ ]:
def precision_recall_at_k(recommended: list, relevant: set, k: int) -> tuple:
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & relevant)
    precision = hits / k if k > 0 else 0
    recall = hits / len(relevant) if relevant else 0
    return precision, recall


def average_precision_at_k(recommended: list, relevant: set, k: int) -> float:
    recommended_k = recommended[:k]
    score, hits = 0.0, 0
    for i, pid in enumerate(recommended_k, start=1):
        if pid in relevant:
            hits += 1
            score += hits / i
    return score / min(len(relevant), k) if relevant else 0.0


def evaluate_hybrid_recommender(eval_df: pd.DataFrame, recommender: "HybridRecommender", K_values=(5, 10, 20)):
    results = {K: {"precision": [], "recall": [], "ap": []} for K in K_values}
    recommended_catalog = set()
    max_k = max(K_values)

    for _, row in eval_df.iterrows():
        input_pid = row["input_product_id"]
        relevant  = row["true_product_ids"]

        recs = recommender.recommend_for_product(input_pid, top_n=max_k)
        recommended_list = recs["product_id"].tolist()
        recommended_catalog.update(recommended_list)

        for K in K_values:
            p, r = precision_recall_at_k(recommended_list, relevant, K)
            ap = average_precision_at_k(recommended_list, relevant, K)
            results[K]["precision"].append(p)
            results[K]["recall"].append(r)
            results[K]["ap"].append(ap)

    summary_rows = []
    for K in K_values:
        summary_rows.append({
            "K": K,
            "Precision@K": round(np.mean(results[K]["precision"]), 4),
            "Recall@K":    round(np.mean(results[K]["recall"]),    4),
            "MAP@K":       round(np.mean(results[K]["ap"]),        4),
        })

    coverage = len(recommended_catalog) / len(products)
    summary_df = pd.DataFrame(summary_rows).set_index("K")
    return summary_df, coverage

In [ ]:
# ── Monta conjunto de avaliação sem carregar order_products__prior.csv inteiro ──

import gc

train_orders = orders.loc[
    orders["eval_set"] == "train",
    ["user_id", "order_id"]
].copy()

prior_orders = orders.loc[
    orders["eval_set"] == "prior",
    ["user_id", "order_id", "order_number"]
].copy()

# Último pedido prior de cada usuário
idx_last_prior = prior_orders.groupby("user_id")["order_number"].idxmax()

last_prior_orders = (
    prior_orders
    .loc[idx_last_prior, ["user_id", "order_id"]]
    .rename(columns={"order_id": "last_prior_order_id"})
)

last_prior_order_ids = set(last_prior_orders["last_prior_order_id"].astype("int32"))

# Lê somente os produtos dos últimos pedidos prior em chunks
prior_chunks = []

for chunk in pd.read_csv(
    ORDER_PRODUCTS_PRIOR_PATH,
    usecols=["order_id", "product_id", "add_to_cart_order"],
    dtype={
        "order_id": "int32",
        "product_id": "int32",
        "add_to_cart_order": "int16",
    },
    chunksize=500_000,
):
    chunk = chunk[chunk["order_id"].isin(last_prior_order_ids)]

    if not chunk.empty:
        prior_chunks.append(chunk)

if prior_chunks:
    last_prior_products = pd.concat(prior_chunks, ignore_index=True)
else:
    last_prior_products = pd.DataFrame(columns=["order_id", "product_id", "add_to_cart_order"])

del prior_chunks
gc.collect()

# Produto de entrada: último produto adicionado no último pedido prior
idx_last_product = last_prior_products.groupby("order_id")["add_to_cart_order"].idxmax()

last_prior_product = (
    last_prior_products
    .loc[idx_last_product, ["order_id", "product_id"]]
    .rename(columns={
        "order_id": "last_prior_order_id",
        "product_id": "input_product_id",
    })
)

last_prior = (
    last_prior_orders
    .merge(last_prior_product, on="last_prior_order_id", how="inner")
    [["user_id", "input_product_id"]]
)

# Ground truth: produtos comprados no pedido train
train_sets = (
    order_products_train
    .merge(train_orders, on="order_id", how="inner")
    .groupby("user_id")["product_id"]
    .apply(set)
    .reset_index()
    .rename(columns={"product_id": "true_product_ids"})
)

eval_df = last_prior.merge(train_sets, on="user_id", how="inner")

eval_sample = eval_df.sample(
    min(1500, len(eval_df)),
    random_state=42
)

del prior_orders, last_prior_orders, last_prior_products, last_prior_product, train_orders
gc.collect()

metrics, coverage = evaluate_hybrid_recommender(
    eval_sample,
    recommender,
    K_values=[5, 10, 20],
)

print("📊 Métricas do Motor Híbrido de Recomendação")
print(metrics.to_string())
print(f"\n📦 Coverage do catálogo: {coverage:.2%}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
x = metrics.index.astype(str)
colors = sns.color_palette("viridis", 3)

axes[0].bar(x, metrics["Precision@K"], color=colors[0])
axes[0].set_title("Precision@K", fontweight="bold"); axes[0].set_xlabel("K")

axes[1].bar(x, metrics["Recall@K"], color=colors[1])
axes[1].set_title("Recall@K", fontweight="bold"); axes[1].set_xlabel("K")

axes[2].bar(x, metrics["MAP@K"], color=colors[2])
axes[2].set_title("MAP@K", fontweight="bold"); axes[2].set_xlabel("K")

plt.suptitle(f"🎯 Avaliação do Motor Híbrido  |  Coverage: {coverage:.1%}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## ⚖️ 7. Comparação entre Modelos

In [ ]:
# ── Compara MB puro, Similaridade pura e Híbrido ──

models = {
    "Market Basket": HybridRecommender(
        market_basket_rules,
        item_item_similarity,
        product_to_idx,
        idx_to_product,
        products,
        weights={"market_basket": 1.0, "similarity": 0.0, "popularity": 0.0},
    ),
    "Híbrido": recommender,
}

if item_item_similarity is not None:
    models["Similaridade"] = HybridRecommender(
        market_basket_rules,
        item_item_similarity,
        product_to_idx,
        idx_to_product,
        products,
        weights={"market_basket": 0.0, "similarity": 1.0, "popularity": 0.0},
    )
else:
    print("⚠️ Modelo 'Similaridade pura' pulado porque item_item_similarity não foi carregada.")

comparison = {}

for label, model in models.items():
    m, cov = evaluate_hybrid_recommender(eval_sample, model, K_values=[10])

    comparison[label] = {
        "Precision@10": m.loc[10, "Precision@K"],
        "Recall@10": m.loc[10, "Recall@K"],
        "MAP@10": m.loc[10, "MAP@K"],
        "Coverage": cov,
    }

comparison_df = pd.DataFrame(comparison).T

print("📊 Comparação entre Modelos (K=10)")
print(comparison_df.round(4).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
comparison_df[["Precision@10","Recall@10","MAP@10"]].plot(kind="bar", ax=ax, color=sns.color_palette("viridis",3))
ax.set_title("⚖️ Comparação de Modelos de Recomendação (K=10)", fontsize=13, fontweight="bold")
ax.set_ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 💾 8. Exportação do Motor

In [ ]:
import pickle

with open(PROCESSED_PATH / "hybrid_recommender.pkl", "wb") as f:
    pickle.dump({
        "weights": recommender.weights,
        "mb_index_sample": dict(list(recommender._mb_index.items())[:1000]),
        "uses_similarity": item_item_similarity is not None,
    }, f)

if "comparison_df" in globals():
    comparison_df.to_csv(PROCESSED_PATH / "model_comparison.csv")
    print("✅ Resultados exportados:")
    print("   · model_comparison.csv")
else:
    print("⚠️ comparison_df ainda não existe. Rode a seção de comparação antes de exportar.")

print("   · hybrid_recommender.pkl")


## 📌 Conclusão

| Modelo | Característica | Quando usar |
|---|---|---|
| Market Basket | Coocorrência direta, alta precisão em pares fortes | Recomendação "comprados juntos" |
| Similaridade Cosseno | Captura padrão agregado de consumo | Produtos substitutos / similares |
| **Híbrido** | Combina ambos + popularidade como fallback | Uso geral em produção |

O motor híbrido oferece o melhor equilíbrio entre Precision, Recall e Coverage, evitando os pontos cegos de cada sinal isolado. O próximo notebook (**06 — Embedding**) adiciona uma camada de representação aprendida via rede neural, capaz de capturar similaridade semântica que os métodos baseados em contagem não alcançam.
